# LACSS GRPC call Demo

This is a small notebook demonstrating LACSS model inference by GRPC call to an inference server




## Setting up the environment

In [ ]:
!pip install biopb

In [ ]:
import cv2
import imageio.v2 as imageio
import matplotlib.pyplot as plt
import numpy as np
import grpc

import biopb.image as proto
from biopb.image.utils import serialize_from_numpy

SERVER = "lacss.biopb.org"

# construct gRPC request based on given image
def construct_request(image: np.ndarray) -> proto.DetectionRequest:
    return proto.DetectionRequest(
        image_data = proto.ImageData(pixels=serialize_from_numpy(image)),
        detection_settings = proto.DetectionSettings(
          scaling_hint = 1.0,
        ),
    )

# convert gRPC reply to a label image
def response_to_label(response: proto.DetectionResponse, image: np.ndarray) -> np.ndarray:
    label = np.zeros(image.shape[:2], dtype="uint8")

    for k, det in enumerate(response.detections):

        polygon = [[p.x, p.y] for p in det.roi.polygon.points]
        polygon = np.round(np.array(polygon)).astype(int)

        cv2.fillPoly(label, [polygon], k + 1)

    return label

## Download some data for testing

In [ ]:
!wget -c https://data.mendeley.com/public-files/datasets/894mmsd9nj/files/568e524f-9a95-45a6-9f80-3619969c2a37/file_downloaded -O images.zip

import zipfile

data_path = 'image_data'
with zipfile.ZipFile('images.zip', "r") as f:
    f.extractall(data_path)

image = imageio.imread("image_data/test/000_img.png")

plt.imshow(image)

## Call GRPC server

In [ ]:
request = construct_request(image)

# call server
with grpc.secure_channel(
      target=SERVER,
      credentials=grpc.ssl_channel_credentials(),
    ) as channel:
    stub = proto.ObjectDetectionStub(channel)
    response = stub.RunDetection(request)

plt.imshow(response_to_label(response, image))